# 03 — Extend Remittance Dataset (Aug 2012 – Nov 2025)

This notebook extends Ghimire's original remittance panel (Aug 2012 – Dec 2023) through Nov 2025 by combining:

| Source | Variable | Period |
|--------|----------|--------|
| NRB Balance of Payments (BPM5) | `remittance` | Jan 2024 – Jul 2024 |
| NRB Monthly Macroeconomic Reports (hardcoded) | `remittance` | Aug 2024 – Nov 2025 |
| NRB Exchange Rate file | `exchange_rate` | Jan 2024 – Nov 2025 |
| World Bank Pink Sheet | `oil_price` | Jan 2024 – Nov 2025 |
| Dept. of Foreign Employment (DOFE) | `dofe_departures` | Jan 2024 – Nov 2025 |

**Note:** Remittance values for Aug 2024 – Nov 2025 are sourced from NRB's published Current Macroeconomic and Financial Situation PDF reports (preliminary figures). FY2024/25 sum verified at Rs. 1,723.27 billion against NRB's published full-year total.

## NRB Remittance Sources (Aug 2024 – Nov 2025)

All values reconstructed by differencing cumulative figures from official NRB monthly reports:

**FY 2024/25 (Aug 2024 – Jul 2025)**
- 1-month report (Sep 2024): Rs. 136.93B → Aug 2024
- 3-month report (Nov 2024): Rs. 407.31B cumul → Oct monthly 144.38B
- 4-month report (Dec 2024): Rs. 521.63B cumul → Nov monthly 114.32B
- 5-month report (Jan 2025): Rs. 640.43B cumul → Dec monthly 118.80B
- 7-month report (Mar 2025): Rs. 900.58B cumul; Sep=126B, Jan=119B, Feb=137B confirmed via myRepublica
- 9-month report (May 2025): Rs. 1,191.31B cumul → Apr monthly 140.00B
- 10-month report (Jun 2025): Rs. 1,356.61B cumul → May monthly 165.30B
- 11-month report (Jul 2025): Jun monthly Rs. 176.32B (cumul = 1,532.93B)
- Full-year report (Aug 2025): FY total Rs. 1,723.27B → Jul monthly 190.34B

**FY 2025/26 (Aug 2025 – Nov 2025)**
- 1-month report (Sep 2025): Aug monthly Rs. 177.41B
- 2-month report (Oct 2025): cumul Rs. 352.08B → Sep monthly 174.67B
- 3-month report (Nov 2025): cumul Rs. 553.31B → Oct monthly 201.23B
- 4-month report (Dec 2025): Nov monthly Rs. 133.80B (confirmed via Himalayan Times, Dec 15 2025)

✅ Sum check FY2024/25: 136.93+126.00+144.38+114.32+118.80+123.15+137.00+150.73+140.00+165.30+176.32+190.34 = **1,723.27B**

In [ ]:
import pandas as pd
import numpy as np

# ── FILE PATHS – adjust if needed ─────────────────────────────────────
EXISTING_FILE = 'remittance_2012_2023_ready.xlsx'   # Ghimire's original panel
BOP_FILE      = 'Balance-of-Payments-BPM5.xlsx'
EXCHANGE_FILE = 'Exchange-rate.xlsx'
OIL_FILE      = 'pink_sheet_monthly.xlsx'
DOFE_FILE     = 'MIgrant-Workers_ (1).xlsx'
OUTPUT_FILE   = '../output/remittance_2012_2025_extended.csv'

## 1. Load Existing Data (Ghimire's Panel)

In [ ]:
existing = pd.read_excel(EXISTING_FILE)
existing['date'] = pd.to_datetime(existing['date'])
print(f"Existing: {existing['date'].iloc[0].strftime('%b %Y')} → "
      f"{existing['date'].iloc[-1].strftime('%b %Y')} ({len(existing)} rows)")
print(f"Columns: {list(existing.columns)}")

## 2. BOP Remittance (Jan 2024 – Jul 2024)

Row 31 of BOP BPM5 sheet contains cumulative remittance. We difference consecutive months to get monthly figures.

In [ ]:
df_bop = pd.read_excel(BOP_FILE, sheet_name='BOP BPM5', header=None)
fy_row  = df_bop.iloc[2].ffill().fillna('')
mon_row = df_bop.iloc[3].fillna('')
months_map = {'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12,
              'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7}

dates, cols = [], []
for c in range(4, df_bop.shape[1]):
    fy  = str(fy_row.iloc[c]).strip().replace(' P','').replace('P','').replace(' R','').replace('R','')
    mon = str(mon_row.iloc[c]).strip()
    if mon in months_map and '/' in fy:
        try:
            sy = int(fy.split('/')[0].strip())
            m  = months_map[mon]
            yr = sy if m >= 8 else sy + 1
            dates.append(pd.Timestamp(year=yr, month=m, day=1))
            cols.append(c)
        except:
            continue

rem_cum = [pd.to_numeric(df_bop.iloc[31, c], errors='coerce') for c in cols]
bop = pd.DataFrame({'date': dates, 'rem_cum': rem_cum})
bop['remittance'] = np.nan
for i in range(len(bop)):
    if bop['date'].iloc[i].month == 8:
        bop.loc[bop.index[i], 'remittance'] = bop['rem_cum'].iloc[i]
    else:
        bop.loc[bop.index[i], 'remittance'] = (bop['rem_cum'].iloc[i]
                                                - bop['rem_cum'].iloc[i-1])

bop_new = (bop[bop['date'].between('2024-01-01', '2024-07-31')]
             [['date', 'remittance']].copy())
print(f"BOP: {bop_new['date'].iloc[0].strftime('%b %Y')} → "
      f"{bop_new['date'].iloc[-1].strftime('%b %Y')} ({len(bop_new)} rows)")
bop_new

## 3. NRB Remittance (Aug 2024 – Nov 2025)

Hardcoded from NRB PDF reports. Values in Rs. Billion × 1000 = Rs. Million (to match BOP file units).

In [ ]:
nrb_data = {
    # FY 2024/25 (Aug 2024 – Jul 2025)
    '2024-08':  136.93 * 1000,   # 1-month NRB report
    '2024-09':  126.00 * 1000,   # 3-month cumul minus Aug minus Oct
    '2024-10':  144.38 * 1000,   # 3-month cumul (407.31) - 2-month (262.93)
    '2024-11':  114.32 * 1000,   # 4-month (521.63) - 3-month (407.31)
    '2024-12':  118.80 * 1000,   # 5-month (640.43) - 4-month (521.63)
    '2025-01':  123.15 * 1000,   # 7-month (900.58) - 5-month (640.43) - Feb (137.00)
    '2025-02':  137.00 * 1000,   # 7-month report, monthly breakdown
    '2025-03':  150.73 * 1000,   # 8-month (1051.31) - 7-month (900.58)
    '2025-04':  140.00 * 1000,   # 9-month (1191.31) - 8-month (1051.31)
    '2025-05':  165.30 * 1000,   # 10-month (1356.61) - 9-month (1191.31)
    '2025-06':  176.32 * 1000,   # 11-month report, monthly figure
    '2025-07':  190.34 * 1000,   # FY total (1723.27) - 11-month cumul (1532.93)
    # FY 2025/26 (Aug 2025 – Nov 2025)
    '2025-08':  177.41 * 1000,   # 1-month report FY2025/26
    '2025-09':  174.67 * 1000,   # 2-month cumul (352.08) - 1-month (177.41)
    '2025-10':  201.23 * 1000,   # 3-month cumul (553.31) - 2-month (352.08)
    '2025-11':  133.80 * 1000,   # 4-month report, confirmed Himalayan Times Dec 15 2025
}

nrb_df = pd.DataFrame([
    {'date': pd.Timestamp(k + '-01'), 'remittance': v}
    for k, v in nrb_data.items()
])

print(f"NRB: {nrb_df['date'].iloc[0].strftime('%b %Y')} → "
      f"{nrb_df['date'].iloc[-1].strftime('%b %Y')} ({len(nrb_df)} rows)")

# FY2024/25 sum check
fy_sum = sum(v for k,v in nrb_data.items() if k <= '2025-07')
print(f"FY2024/25 sum: {fy_sum/1000:.2f}B (should be 1723.27B) {'✓' if abs(fy_sum/1000 - 1723.27) < 0.1 else '✗'}")
nrb_df

## 4. Exchange Rate (NRB)

In [ ]:
df_ex = pd.read_excel(EXCHANGE_FILE, sheet_name='Time series',
                      header=None, skiprows=4)
df_ex.columns = ['fiscal_year','month','me_buy','me_sell','me_mid',
                  'avg_buy','avg_sell','avg_mid']
df_ex['fiscal_year'] = df_ex['fiscal_year'].ffill()
df_ex = df_ex[~df_ex['month'].astype(str).str.contains('Annual|Quarter', na=False)]
df_ex = df_ex.dropna(subset=['month'])

full_months = {'August':8,'September':9,'October':10,'November':11,'December':12,
               'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,'July':7}

def parse_ex_date(row):
    mon = str(row['month']).strip()
    fy  = str(row['fiscal_year']).strip()
    if mon not in full_months or '/' not in fy:
        return None
    try:
        sy = int(fy.split('/')[0].strip())
        m  = full_months[mon]
        yr = sy if m >= 8 else sy + 1
        return pd.Timestamp(year=yr, month=m, day=1)
    except:
        return None

df_ex['date'] = df_ex.apply(parse_ex_date, axis=1)
df_ex['exchange_rate'] = pd.to_numeric(df_ex['avg_mid'], errors='coerce')
ex_series = df_ex[['date', 'exchange_rate']].dropna()
print(f"Exchange rate: {ex_series['date'].iloc[0].strftime('%b %Y')} → {ex_series['date'].iloc[-1].strftime('%b %Y')}")

## 5. Oil Price (World Bank Pink Sheet)

In [ ]:
df_oil = pd.read_excel(OIL_FILE, sheet_name='Monthly Prices',
                       header=None, skiprows=6)
df_oil.columns = ['date'] + [f'c{i}' for i in range(1, len(df_oil.columns))]
df_oil = df_oil.rename(columns={'c2': 'oil_price'})
df_oil['date'] = df_oil['date'].astype(str)
df_oil = df_oil[df_oil['date'].str.match(r'\d{4}M\d{2}')]
df_oil['date'] = pd.to_datetime(df_oil['date'], format='%YM%m')
df_oil['oil_price'] = pd.to_numeric(df_oil['oil_price'], errors='coerce')
oil_series = df_oil[['date', 'oil_price']].dropna()
print(f"Oil price: {oil_series['date'].iloc[0].strftime('%b %Y')} → {oil_series['date'].iloc[-1].strftime('%b %Y')}")

## 6. DOFE Departures

In [ ]:
df_dofe = pd.read_excel(DOFE_FILE, sheet_name='Migrant Worker', header=1)
df_dofe.columns = ['date','new_entry','renew_entry','total_outflow',
                   'cum_new','cum_renew','cum_total']
df_dofe = df_dofe.dropna(subset=['date'])
df_dofe = df_dofe[df_dofe['date'].astype(str).str.match(r'\d{4}-\d{2}')]
df_dofe['date'] = pd.to_datetime(df_dofe['date']).dt.to_period('M').dt.to_timestamp()
df_dofe['dofe_departures'] = pd.to_numeric(df_dofe['total_outflow'], errors='coerce').astype(int)
dofe_series = df_dofe[['date', 'dofe_departures']].dropna()
print(f"DOFE: {dofe_series['date'].iloc[0].strftime('%b %Y')} → {dofe_series['date'].iloc[-1].strftime('%b %Y')}")

## 7. Build New Rows and Merge (Jan 2024 – Nov 2025)

In [ ]:
new_dates = pd.date_range('2024-01-01', '2025-11-01', freq='MS')
new_df = pd.DataFrame({'date': new_dates})

all_rem = pd.concat([bop_new, nrb_df], ignore_index=True)
new_df = new_df.merge(all_rem, on='date', how='left')
new_df = new_df.merge(ex_series, on='date', how='left')
new_df = new_df.merge(oil_series, on='date', how='left')
new_df = new_df.merge(dofe_series, on='date', how='left')

print(f"Missing remittance: {new_df['remittance'].isna().sum()} rows")
new_df

## 8. Append to Existing Data and Save

In [ ]:
for col in existing.columns:
    if col not in new_df.columns:
        new_df[col] = np.nan
new_df = new_df[existing.columns]

extended = pd.concat([existing, new_df], ignore_index=True)
extended = extended.sort_values('date').reset_index(drop=True)
extended['date'] = extended['date'].dt.date   # clean date, no timestamp

print(f"Extended: {str(extended['date'].iloc[0])} → {str(extended['date'].iloc[-1])} ({len(extended)} rows)")
print(f"Missing remittance: {extended['remittance'].isna().sum()}")
extended.tail(25)

In [ ]:
extended.to_csv(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")

## 9. Validation Checks

In [ ]:
# Check 1: Row count
print(f"Rows: {len(extended)} (expected 160)")

# Check 2: FY2024/25 sum
fy_check = extended[extended['date'].astype(str).between('2024-08-01','2025-07-01')]['remittance'].sum()
print(f"FY2024/25 sum: {fy_check/1000:.2f}B (expected 1723.27B) {'✓' if abs(fy_check/1000-1723.27)<1 else '✗'}")

# Check 3: Unit consistency at join point
jul24 = extended[extended['date'].astype(str)=='2024-07-01']['remittance'].values[0]
aug24 = extended[extended['date'].astype(str)=='2024-08-01']['remittance'].values[0]
ratio = aug24/jul24
print(f"Jul 2024: {jul24:,.0f} | Aug 2024: {aug24:,.0f} | ratio: {ratio:.2f} {'✓' if 0.5<ratio<3 else '✗ UNIT MISMATCH'}")

# Check 4: No missing values
print(f"Missing values:\n{extended.isnull().sum()}")